In [ ]:
root = "M://Diploma/experiments/vesuvius/fragments"

### Data loading

For the baseline, we will do simple per-pixel classification. That is, we do not account for the neighborhood in any way. The only input we use are the raw pixel values from IR/CT.

Since fitting all 65 CT layers into RAM is infeasible (takes up more than 100 GB), here we will also select the most useful of those for further experiments, so that we do not have to worry about implementing models in external memory.

In [ ]:
import numpy
import typing
import warnings
import PIL.Image

warnings.filterwarnings("ignore", category=PIL.Image.DecompressionBombWarning)

def load_fragment(fragment: int, depths: typing.Optional[typing.List[int]]):
    mask = numpy.array(PIL.Image.open(f"{root}/Frag{fragment}/mask.png")) > 0
    pixels = numpy.flatnonzero(mask)
    ink = numpy.array(PIL.Image.open(f"{root}/Frag{fragment}/inklabels.png")).ravel()[pixels] > 0

    if depths is None:
        ir = numpy.array(PIL.Image.open(f"{root}/Frag{fragment}/ir.png")).ravel()[pixels]
        return numpy.expand_dims(ir, 1), ink

    ct = numpy.empty((len(pixels), len(depths)), dtype=numpy.float32)
    for i, depth in enumerate(depths):
        ct[:, i] = numpy.array(PIL.Image.open(f"{root}/Frag{fragment}/surface_volume/{depth:02d}.tif")).ravel()[pixels]
    return ct, ink

def load_fragments(fragments, depths: typing.Optional[typing.List[int]]):
    values = list(load_fragment(fragments[0], depths))
    for fragment in fragments[1:]:
        fragment_values = load_fragment(fragment, depths)
        for i in range(len(values)):
            values[i] = numpy.concatenate([ values[i], fragment_values[i] ])
    return values

### Evaluation metrics

The main metric used in the competition is the $F_{0.5}$ score defined as follows:
$$ F_{0.5} = \dfrac{1.25 pr}{0.25 p + r} = \dfrac{1.25 TP}{1.25TP + FP + 0.25FN} $$
This metric scores precision higher than recall, which is designed to improve the ability to form coherent characters out of detected ink areas.

Additionally, for a more comprehensive evaluation, we compute the other typical segmentation metrics: per-pixel accuracy, precison, recall, F1-score, ROC-AUC, and Jaccard score.

For the leaderboard, the competition uses additional private data, which is not available in this project. To circumvent this, we will use Fragment 4 as a held-out test object used for evaluation. The results should generally reflect the scores in the competition, although direct comparison to the metrics reported there is not possible.

In [ ]:
import numpy
import sklearn.metrics
import matplotlib.pyplot as plt

def estimate_quality(pred_proba: numpy.ndarray, masks: numpy.ndarray) -> dict:
    scores = pred_proba.flatten()
    preds = (scores >= 0.5).astype(numpy.int8)
    targets = masks.flatten().astype(numpy.int8)
    return {
        "F0.5": sklearn.metrics.fbeta_score(targets, preds, beta=0.5, zero_division=0),
        "Accuracy": sklearn.metrics.accuracy_score(targets, preds),
        "AUC-ROC": sklearn.metrics.roc_auc_score(targets, scores),
        "Precision": sklearn.metrics.precision_score(targets, preds, zero_division=0),
        "Recall": sklearn.metrics.recall_score(targets, preds, zero_division=0),
        "F1-score": sklearn.metrics.f1_score(targets, preds, zero_division=0),
        "Jaccard score": sklearn.metrics.jaccard_score(targets, preds, zero_division=0),
    }

def plot_prediction(pred_proba: numpy.ndarray, fragment: int, ax):
    mask = numpy.array(PIL.Image.open(f"{root}/Frag{fragment}/mask.png")) > 0
    prediction = numpy.zeros(mask.shape, dtype=bool)
    prediction[mask] = pred_proba.ravel() >= 0.5

    if ax is None:
        target = numpy.array(PIL.Image.open(f"{root}/Frag{fragment}/inklabels.png")) > 0
        fig, axes = plt.subplots(1, 2, figsize=(4, 3))
        axes[0].imshow(target, cmap="gray", vmin=0, vmax=1)
        axes[0].set_title("Ink target")
        axes[1].imshow(prediction, cmap="gray", vmin=0, vmax=1)
        axes[1].set_title("Predicted ink")
        for ax in axes:
            ax.axis("off")
        fig.tight_layout()
        plt.show()
        plt.close(fig)
    else:
        ax.imshow(prediction, cmap="gray", vmin=0, vmax=1)
        ax.set_title("Predicted ink")
        ax.axis("off")

In all the following experiments, we explore two predictions: one formed from the infrared scans, and one from the 65 layers of the CT scans. The final evaluation matching the competition is only the latter one, yet the easier IR task is a simpler one that might give us some insight into the dataset and help devise a model for the harder part.

For this notebook, we also perform multiple CT experiments with different subsets of layers to find the ones that have no useful information and can be discarded from further experiments for performance reasons.
- Every second layer
- Every third layer
- Every fourth layer
- Layers 0 - 15: inner layers of the papyrus. We expect they are not useful.
- Layers 20 - 40: the writing edge. We expect they are the most useful.
- Layers 45 - 60: outer layers. We expect they are not useful.
- Layers 15 - 45: all layers except those that we assume are not uesful

In [ ]:
experiments = {
    "IR": None,
    "CT: every 2nd layer": list(range(0, 65, 2)),
    "CT: every 3rd layer": list(range(0, 65, 3)),
    "CT: every 4th layer": list(range(0, 65, 4)),
    "CT: layers 0-15": list(range(0, 16)),
    "CT: layers 20-40": list(range(20, 41)),
    "CT: layers 45-64": list(range(45, 65)),
    "CT: layers 15-45": list(range(15, 46)),
}

### Dummy classifiers

First, let's try the simplest models that just predict constants.

In [ ]:
import sklearn.dummy

def try_dummy(model: sklearn.dummy.DummyClassifier):
    train_X, train_y = load_fragments([1, 2, 3], depths=None)
    test_X, test_y = load_fragments([4], depths=None)

    model.fit(train_X, train_y)
    preds = model.predict_proba(test_X)[:, 1]
    scores = estimate_quality(preds, test_y)
    plot_prediction(preds, fragment=4, ax=None)
    return scores

In [ ]:
try_dummy(sklearn.dummy.DummyClassifier(strategy="constant", constant=0))

In [ ]:
try_dummy(sklearn.dummy.DummyClassifier(strategy="constant", constant=1))

Expectdly, the $F_{0.5}$ score is zero. However, the class imbalance here is obvious: only 10% of pixels are labeled as having inc.

### Basic models

In [ ]:
import catboost
import sklearn.tree
import sklearn.ensemble
import sklearn.linear_model
import sklearn.preprocessing

models = {
    "Decision stump": sklearn.tree.DecisionTreeClassifier(max_depth=1, random_state=42),
    "Logistic regression (C=0.1)": sklearn.linear_model.LogisticRegression(C=0.1, random_state=42, n_jobs=-1),
    "Logistic regression (C=1.0)": sklearn.linear_model.LogisticRegression(C=1.0, random_state=42, n_jobs=-1),
    "Logistic regression (C=10.0)": sklearn.linear_model.LogisticRegression(C=10.0, random_state=42, n_jobs=-1),
    "Decision tree (depth=5)": sklearn.tree.DecisionTreeClassifier(max_depth=5, random_state=42),
    "Decision tree (depth=10)": sklearn.tree.DecisionTreeClassifier(max_depth=10, random_state=42),
    "Decision tree (depth=25)": sklearn.tree.DecisionTreeClassifier(max_depth=25, random_state=42),
    "Random forest": sklearn.ensemble.RandomForestClassifier(max_depth=10, max_samples=0.05, random_state=42, n_jobs=1, verbose=0),
    "Gradient boosting": sklearn.ensemble.HistGradientBoostingClassifier(max_depth=4, random_state=42, verbose=0),
    "CatBoost": catboost.CatBoostClassifier(iterations=100, depth=4, random_seed=42, verbose=False),
}

In [ ]:
def evaluate(model_name: str, *data):
    print(f'    Started {model_name}')
    train_X, train_y, test_X, test_y = data
    model = sklearn.base.clone(models[model_name]).fit(train_X, train_y)
    print(f'    Finished {model_name}')
    return estimate_quality(model.predict_proba(test_X)[:, 1], test_y)

runs = []
for experiment, depths in experiments.items():
    print(f'Running {experiment}')
    train_X, train_y = load_fragments([1, 2, 3], depths=depths)
    test_X, test_y = load_fragments([4], depths=depths)

    ss = sklearn.preprocessing.StandardScaler(copy=False)
    train_X = ss.fit_transform(train_X)
    test_X = ss.transform(test_X)

    for name, model in models.items():
        result = evaluate(name, train_X, train_y, test_X, test_y)
        runs.append({'Model': name, 'Experiment': experiment, **result})
        print(f"    {name}: F0.5={result['F0.5']:.4f}")

In [ ]:
import pandas

results = pandas.DataFrame(runs)
results.to_csv("02_baseline_results.csv")
results